# Density-matrix screening strategies

QuantumXCT prunes candidate gates with the density-matrix difference $\Delta\rho$ (co-culture
minus mono-culture). This benchmark checks that cheaper ways of computing that screen give
**the same candidate list** as the reference, and how much time and memory they save:

| Strategy | Computation |
|---|---|
| `reference` | `find_cnot_candidates_from_state_diff`, complex128 (pipeline default) |
| `dense(float32)` / `dense(float16)` | `find_cnot_candidates_dense` at lower precision (float16 = stress test) |
| `sparse(float32)` | `find_cnot_candidates_sparse`, support-block version that scales to larger panels |

It runs the two scripts next to this notebook, which stay the single source of truth:

1. `test_dm_strategies.py`: equivalence check on a small synthetic system (no data needed).
2. `benchmark_dm_strategies.py`: the benchmark on the ovarian data
   (needs `dataset_co_mo2.h5ad`; build it with `00_build_datasets.ipynb`).

Results: `results/dm_strategies/dm_strategies_benchmark.csv` and `.tex`.

In [ ]:
# Run from the QuantumXCT/python root, wherever Jupyter was started, so the data
# are found and every output goes to python/results/.
import os, sys, subprocess
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "qxct")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("working directory:", _root)

HERE = os.path.join("benchmarks", "dm_strategies")
QUALITY = False      # True: also run the topology search + angle optimization per lossy strategy (slower)


def run(script, *args):
    """Run one of the benchmark scripts with this Python and show its output."""
    p = subprocess.run([sys.executable, os.path.join(HERE, script), *args],
                       capture_output=True, text=True)
    print(p.stdout)
    if p.returncode != 0:
        print(p.stderr[-3000:])
        raise RuntimeError(f"{script} failed (exit {p.returncode})")

## 1. Equivalence check (synthetic, 6 qubits)

Every strategy should return exactly the candidate set of the current pipeline
(`match_current = True`).

In [ ]:
run("test_dm_strategies.py")

## 2. Ovarian benchmark

In [ ]:
if not os.path.exists("dataset_co_mo2.h5ad"):
    raise FileNotFoundError("dataset_co_mo2.h5ad not found: run 00_build_datasets.ipynb first")
run("benchmark_dm_strategies.py", *(["--quality"] if QUALITY else []))

In [ ]:
import pandas as pd
df = pd.read_csv(os.path.join("results", "dm_strategies", "dm_strategies_benchmark.csv"))
df

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
col = ["#9370DB" if s.startswith("reference") else "#56B4E9" for s in df["strategy"]]
for ax, c, lab in zip(axes, ["time_ms", "peak_KB", "overlap_pct"], ["wall time (ms)", "peak memory (KB)", "overlap with reference (%)"]):
    ax.bar(df["strategy"], df[c], color=col)
    ax.set_title(lab); ax.tick_params(axis="x", rotation=30)
axes[2].set_ylim(0, 105)
fig.tight_layout(); fig.savefig(os.path.join("results", "dm_strategies", "dm_strategies_benchmark.png"), dpi=140)
plt.show()

**Reading the table.** `matches_reference` and `overlap_pct = 100` mean the strategy finds exactly
the reference candidates; `jaccard` is the set similarity; `max_signal` is the largest
$\Delta\rho$ peak; `time_ms` and `peak_KB` are wall time and peak memory. With `QUALITY = True`,
lossy strategies also report `n_gates`, `final_kl` and `kl_delta_vs_ref` after a full search.